# Lesson 03 — Video → Frames

Before we can embed a video with CLIP, we need to break it into individual **frames** (images).

This lesson:
1. Uploads a sample video to S3
2. Runs a Batch job that extracts frames and saves them back to S3
3. Downloads a few frames and shows them here

### What is a frame?
A video at 30 fps has 30 images per second. Each image = 1 frame. A 60-second video = 1,800 frames.
We don't need all of them — we'll extract 1 every 30 frames (= 1 per second).

## Step 1 — Check .env

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../.env")

S3_BUCKET = os.environ["S3_BUCKET"]
print(f"S3 bucket : {S3_BUCKET}")
print(f"Job queue : {os.environ['BATCH_JOB_QUEUE']}")

## Step 2 — Upload video & submit Batch job

`submit_job.py` does two things:
- Uploads `assets/sample.mp4` to `s3://<bucket>/videos/sample.mp4`
- Submits the extraction job to Batch, passing `S3_BUCKET` and `VIDEO_KEY` as env vars

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "submit_job.py", "--video", "assets/sample.mp4", "--every-n", "30"],
)
print("Exit code:", result.returncode)

## Step 3 — List extracted frames in S3

In [ ]:
import boto3

s3     = boto3.client("s3")
prefix = "frames/sample/"

resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=20)
keys = [obj["Key"] for obj in resp.get("Contents", [])]

print(f"Found {len(keys)} frames (showing first 5):")
for k in keys[:5]:
    print(f"  {k}")

## Step 4 — Download and display 6 frames

In [ ]:
import io
import matplotlib.pyplot as plt
from PIL import Image

# Pick 6 evenly spaced frames to show
step      = max(1, len(keys) // 6)
to_show   = keys[::step][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes      = axes.flatten()

for ax, key in zip(axes, to_show):
    obj  = s3.get_object(Bucket=S3_BUCKET, Key=key)
    img  = Image.open(io.BytesIO(obj["Body"].read()))
    ax.imshow(img)
    ax.set_title(os.path.basename(key), fontsize=8)
    ax.axis("off")

plt.suptitle("Extracted frames from sample.mp4", fontsize=12)
plt.tight_layout()
plt.show()

## Key Takeaway

> The S3 ↔ Batch pattern: **download input from S3 → process → upload results to S3**.
> Every Batch job in this course follows this pattern. Learn it once, use it everywhere.

---

## Next lesson → [04 — Frames → Embeddings](../04-frames-to-embeddings/notebook.ipynb)

We'll run CLIP on each frame to turn images into vectors — this is where the GPU does real work.